In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
# keras.layers.Dense
# keras.models.Sequential
# tf.keras.optimizers.RMSprop(0.001)


In [ ]:
tf.keras.utils.set_random_seed(42)

dataset = pd.read_csv('data.csv')
dataset.tail()

dataset = dataset.drop(columns=['Name']).dropna()
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

train_labels = train_dataset.pop('Price')
test_labels = test_dataset.pop('Price')

train_stats = train_dataset.describe()
train_stats = train_stats.transpose()
train_stats

def norm(x):
    return (x - train_stats['mean']) / train_stats['std']

normed_train_data = norm(train_dataset)
normed_test_data = norm(test_dataset)

def build_model():
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=[len(train_dataset.keys())]),
        layers.Dense(64, activation='relu'),
        layers.Dense(1)
    ])

    optimizer = tf.keras.optimizers.RMSprop(0.001)

    model.compile(loss='mse',
                  optimizer=optimizer,
                  metrics=['mae', 'mse'])
    return model

model = build_model()

history = model.fit(
    normed_train_data,
    train_labels,
    epochs=250,
    validation_split=0.2,
    verbose=0,
)

hist = pd.DataFrame(history.history)
hist['epoch'] = history.epoch
hist.tail()

def plot_history(history):
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch

    plt.figure(figsize=(8, 5))
    plt.xlabel('Epoch')
    plt.ylabel('Mean Abs Error')
    plt.plot(hist['epoch'], hist['mae'], label='Train Error')
    plt.plot(hist['epoch'], hist['val_mae'], label='Val Error')
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.xlabel('Epoch')
    plt.ylabel('Mean Square Error')
    plt.plot(hist['epoch'], hist['mse'], label='Train Error')
    plt.plot(hist['epoch'], hist['val_mse'], label='Val Error')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(history)

loss, mae, mse = model.evaluate(normed_test_data, test_labels, verbose=0)
print('Testing set Mean Abs Error: {:5.2f}'.format(mae))
print('Testing set Mean Squared Error: {:5.2f}'.format(mse))

best_mae_index = int(hist['val_mae'].idxmin())
best_mae_epoch = best_mae_index + 1
last_train_mae = float(hist['mae'].iloc[-1])
last_val_mae = float(hist['val_mae'].iloc[-1])
best_val_mae = float(hist['val_mae'].iloc[best_mae_index])

print(f'Najbolja epoha prema validacionoj MAE: {best_mae_epoch}')
print(f'Najbolja validaciona MAE: {best_val_mae:.4f}')
print(f'Poslednja trening MAE: {last_train_mae:.4f}')
print(f'Poslednja validaciona MAE: {last_val_mae:.4f}')

if last_val_mae > best_val_mae * 1.05 and last_train_mae < float(hist['mae'].iloc[best_mae_index]):
    print('Model pokazuje znake preprilagodjavanja.')
else:
    print('Na osnovu validacione greske nema jakog znaka preprilagodjavanja.')
